# 🛡️ کنترل کیفیت در Climatology Engine

این نوت‌بوک سیستم Quality Flag را معرفی می‌کند که کیفیت برازش توزیع‌ها را ارزیابی می‌کند.

**مواردی که یاد می‌گیرید:**
- مفهوم Quality Flag در برازش توزیع‌ها
- انواع فلگ‌های کیفیت (PASS، LOW_SAMPLE، NO_CONVERGENCE، OUTLIER، HIGH_AICC، BAD_SKEW)
- نحوه ارزیابی کیفیت برازش
- تنظیم آستانه‌ها (thresholds) برای هر فلگ
- نمایش کیفیت برازش برای مدل‌های مختلف
- تفسیر نتایج Quality Flag

---

## 📐 معرفی سیستم Quality Flag

سیستم Quality Flag به‌صورت خودکار کیفیت هر برازش را ارزیابی کرده و فلگ‌های مناسب را اختصاص می‌دهد.

### فلگ‌های کیفیت

| کد | نام | توضیح |
|----|-----|-------|
| ۰ | PASS | برازش موفقیت‌آمیز |
| ۱ | LOW_SAMPLE | تعداد داده‌ها کمتر از حداقل مجاز (۳) |
| ۲ | NO_CONVERGENCE | بهینه‌سازی همگرا نشده است |
| ۳ | OUTLIER | وجود نقاط پرت در داده |
| ۴ | HIGH_AICC | AICc از آستانه مجاز بالاتر است |
| ۵ | BAD_SKEW | چولگی خارج از محدوده مجاز است |
| ۶ | NAN_INPUT | داده شامل NaN است |
| ۷ | INF_INPUT | داده شامل Inf است |

### آستانه‌های پیش‌فرض

| پارامتر | مقدار پیش‌فرض | توضیح |
|---------|---------------|-------|
| `min_sample_size` | ۳ | حداقل تعداد داده برای برازش |
| `threshold_aicc` | ۱۰۰۰ | حداکثر AICc مجاز |
| `threshold_skew` | ۵.۰ | حداکثر چولگی مطلق مجاز |
| `outlier_sigma` | ۴.۰ | آستانه سیگما برای تشخیص نقاط پرت |

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins
from core.quality.quality_flag import QualityFlag

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ کتابخانه‌ها بارگذاری شدند.')

In [ ]:
# نمایش فلگ‌های کیفیت
flag_names = {
    QualityFlag.PASS: '✅ PASS',
    QualityFlag.LOW_SAMPLE: '⚠️ LOW_SAMPLE',
    QualityFlag.NO_CONVERGENCE: '❌ NO_CONVERGENCE',
    QualityFlag.OUTLIER: '⚠️ OUTLIER',
    QualityFlag.HIGH_AICC: '⚠️ HIGH_AICC',
    QualityFlag.BAD_SKEW: '⚠️ BAD_SKEW',
    QualityFlag.NAN_INPUT: '❌ NAN_INPUT',
    QualityFlag.INF_INPUT: '❌ INF_INPUT',
}

print("📋 فلگ‌های کیفیت:")
print("=" * 50)
for code, name in flag_names.items():
    print(f"   {code}: {name}")
print("=" * 50)

In [ ]:
# بارگذاری پلاگین‌های توزیع
plugins = load_plugins()
print(f'✅ تعداد توزیع‌های بارگذاری شده: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# بارگذاری داده نمونه
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# انتخاب داده tmean برای یک سال
data_year = data[:365, 1]

print(f'📊 تعداد داده‌ها: {len(data_year)}')
print(f'   میانگین: {np.mean(data_year):.2f}°C')
print(f'   انحراف معیار: {np.std(data_year):.2f}°C')

In [ ]:
# برازش همه توزیع‌ها و ارزیابی کیفیت
quality_results = []
print("\n🔄 در حال برازش و ارزیابی کیفیت...\n")

for name, dist in distributions.items():
    try:
        res = dist.fit(data_year)
        flags = QualityFlag.evaluate(res, data_year)
        quality_results.append({
            'توزیع': name,
            'AICc': res.get('aicc', np.nan),
            'فلگ‌ها': flags,
            'تعداد فلگ': len(flags),
            'وضعیت': 'PASS' if QualityFlag.PASS in flags else 'FAIL'
        })
        flag_str = ', '.join([flag_names.get(f, str(f)) for f in flags])
        print(f"{'✅' if QualityFlag.PASS in flags else '⚠️'} {name}: {flag_str}")
    except Exception as e:
        print(f"❌ {name}: خطا - {str(e)}")

print("\n✅ ارزیابی کیفیت کامل شد.")

In [ ]:
# ایجاد جدول کیفیت
quality_df = pd.DataFrame(quality_results)
quality_df['تعداد فلگ'] = quality_df['فلگ‌ها'].apply(len)
quality_df = quality_df.sort_values('AICc').reset_index(drop=True)
quality_df.index = quality_df.index + 1

# نمایش جدول
print("📊 جدول کیفیت برازش مدل‌ها:")
print("=" * 80)
display_cols = ['توزیع', 'AICc', 'تعداد فلگ', 'وضعیت']
quality_df[display_cols]

In [ ]:
# نمایش فلگ‌های هر مدل به‌صورت جزئی
print("📋 جزئیات فلگ‌های کیفیت:")
print("=" * 80)
for _, row in quality_df.iterrows():
    flags = row['فلگ‌ها']
    flag_str = ', '.join([flag_names.get(f, str(f)) for f in flags])
    print(f"\n{row['توزیع']} ({row['وضعیت']}):")
    print(f"   AICc: {row['AICc']:.2f}")
    print(f"   فلگ‌ها: {flag_str if flags else 'هیچ'}")

In [ ]:
# تابع برای نمایش توزیع فلگ‌ها
def get_flag_counts(quality_df):
    """محاسبه تعداد هر نوع فلگ"""
    all_flags = []
    for flags in quality_df['فلگ‌ها']:
        all_flags.extend(flags)
    counts = {}
    for f in all_flags:
        counts[f] = counts.get(f, 0) + 1
    return counts

flag_counts = get_flag_counts(quality_df)

print("📊 توزیع فلگ‌های کیفیت:")
print("=" * 50)
for code, count in sorted(flag_counts.items()):
    name = flag_names.get(code, f'UNKNOWN_{code}')
    print(f"   {name}: {count}")
print("=" * 50)

In [ ]:
# رسم نمودار توزیع فلگ‌ها
if flag_counts:
    fig, ax = plt.subplots(figsize=(10, 6))

    labels = [flag_names.get(code, f'UNKNOWN_{code}') for code in flag_counts.keys()]
    values = list(flag_counts.values())

    colors = ['#2ecc71' if code == 0 else '#e74c3c' if code in [1, 2, 6, 7] else '#f39c12' 
              for code in flag_counts.keys()]

    bars = ax.bar(labels, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1)

    ax.set_xlabel('نوع فلگ', fontsize=12)
    ax.set_ylabel('تعداد', fontsize=12)
    ax.set_title('توزیع فلگ‌های کیفیت در بین مدل‌ها', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()

In [ ]:
# بررسی مدل‌های PASS و FAIL
pass_models = quality_df[quality_df['وضعیت'] == 'PASS']
fail_models = quality_df[quality_df['وضعیت'] == 'FAIL']

print(f"✅ مدل‌های PASS: {len(pass_models)}")
if len(pass_models) > 0:
    print(f"   {', '.join(pass_models['توزیع'].tolist())}")

print(f"\n❌ مدل‌های FAIL: {len(fail_models)}")
if len(fail_models) > 0:
    print(f"   {', '.join(fail_models['توزیع'].tolist())}")

In [ ]:
# تنظیم آستانه‌های جدید و ارزیابی مجدد
print("\n🔧 ارزیابی مجدد با آستانه‌های جدید:")
print("   - min_sample_size: 5")
print("   - threshold_aicc: 500")
print("   - threshold_skew: 3.0")
print("   - outlier_sigma: 3.0")

quality_results_new = []
for name, dist in distributions.items():
    try:
        res = dist.fit(data_year)
        flags = QualityFlag.evaluate(res, data_year, threshold_aicc=500)
        quality_results_new.append({
            'توزیع': name,
            'AICc': res.get('aicc', np.nan),
            'فلگ‌ها': flags,
            'وضعیت': 'PASS' if QualityFlag.PASS in flags else 'FAIL'
        })
    except:
        pass

quality_df_new = pd.DataFrame(quality_results_new)

# مقایسه نتایج قبل و بعد
pass_old = len(quality_df[quality_df['وضعیت'] == 'PASS'])
pass_new = len(quality_df_new[quality_df_new['وضعیت'] == 'PASS'])

print(f"\n📊 مقایسه نتایج:")
print(f"   قبل از تغییر آستانه‌ها: {pass_old} مدل PASS")
print(f"   بعد از تغییر آستانه‌ها: {pass_new} مدل PASS")

## 📋 جمع‌بندی

در این نوت‌بوک یاد گرفتید:

✅ سیستم Quality Flag و انواع فلگ‌ها
✅ نحوه ارزیابی کیفیت برازش با `QualityFlag.evaluate()`
✅ نمایش فلگ‌های کیفیت برای هر مدل
✅ تحلیل توزیع فلگ‌ها در بین مدل‌ها
✅ تأثیر تنظیم آستانه‌ها بر نتایج کیفیت

---

**نکات کلیدی:**

1. فلگ `PASS` به معنای برازش موفقیت‌آمیز است.
2. فلگ‌های `LOW_SAMPLE` و `NAN_INPUT` نشان‌دهنده مشکل در داده هستند.
3. فلگ `HIGH_AICC` نشان‌دهنده کیفیت پایین مدل است.
4. آستانه‌ها را می‌توان با توجه به نیاز پروژه تنظیم کرد.
5. مدل‌های با فلگ `PASS` قابل اعتمادتر هستند.

---

**مراحل بعدی:**
- نوت‌بوک ۰۶: عدم‌قطعیت Bootstrap
- نوت‌بوک ۰۷: پردازش موازی
- نوت‌بوک ۰۸: مصورسازی داده